# 🏆 Camada Gold — Consagração do Negócio & Modelagem Dimensional

## 📌 Objetivos da Etapa
Esta etapa é responsável por transformar os dados limpos e padronizados da **Camada Silver** em um ecossistema de **Data Marts operacionais** e um modelo **Star Schema (Esquema Estrela)** pronto para consumo de Business Intelligence (BI) e Analytics.

### 📐 Arquitetura da Modelagem Dimensional
- **Tabela Fato (`fact_movies_performance`):** Consolida as métricas financeiras (em USD e BRL convertidos na data de lançamento) e de engajamento no grão único de 1 registro por filme.
- **Tabelas de Dimensão:** Armazenam os atributos descritivos e metadados (`dim_movies`, `dim_genres`, `dim_people`, `dim_companies`, `dim_reviews`).
- **Tabelas Ponte (Bridge Tables):** Permitem relacionamentos N:N (Muitos-para-Muitos) entre os filmes e dimensões periféricas (gêneros, elenco/equipe e produtoras) sem duplicar o grão da Fato.
- **Chaves Substitutas (Surrogate Keys - SK):** Geradas de forma sequencial determinística para isolar o modelo de mudanças nos IDs originais.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, expr, row_number, sha2, concat_ws, coalesce, 
    avg, count, round, explode, split, trim, current_timestamp
)

# Schemas do Unity Catalog
silver_schema = "workspace.cinedata_silver"
gold_schema = "workspace.cinedata_gold"

# Garantir a criação do schema Gold
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

DataFrame[]

In [0]:
# Listar todas as tabelas do schema Silver
display(spark.sql("SHOW TABLES IN workspace.cinedata_silver"))

database,tableName,isTemporary
cinedata_silver,tb_avaliacoes_usuarios,false
cinedata_silver,tb_cotacao_dolar,false
cinedata_silver,tb_financeiro_filmes,false
cinedata_silver,tb_generos,false
cinedata_silver,tb_info_filmes,false
cinedata_silver,tb_metricas_engajamento,false
cinedata_silver,tb_pessoas_empresas,false


---
## 1️. Construção das Tabelas de Dimensão

Nesta célula, construímos o catálogo descritivo do Data Warehouse. Cada dimensão recebe uma **Surrogate Key (SK)** baseada na função `row_number()` para garantir chave primária única e inteira (`BIGINT`).

- **`dim_movies`:** Metadados e detalhes dos filmes.
- **`dim_genres`:** Catálogo único e deduplicado de gêneros cinematográficos.
- **`dim_people`:** Elenco e equipe técnica ('Ator', 'Diretor', 'Roteirista').
- **`dim_companies`:** Produtoras e estúdios cinematográficos.
- **`dim_reviews`:** Consolidação sumarizada das avaliações dos usuários (total de votos e média arredondada).

In [0]:
# 1. dim_movies
df_silver_movies = spark.table(f"{silver_schema}.tb_info_filmes")
col_id_movie = "id_filme" if "id_filme" in df_silver_movies.columns else "id"

df_dim_movies = df_silver_movies.select(
    row_number().over(Window.orderBy(col_id_movie)).cast("bigint").alias("sk_movie_id"),
    col(col_id_movie).alias("id_filme"),
    col("titulo"),
    col("data_lancamento"),
    col("ano_lancamento"),
    col("duracao_minutos"),
    col("idioma_original"),
    col("status_filme"),
    col("sinopse")
).dropDuplicates(["id_filme"])

df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_movies")

# 2. dim_genres
df_silver_genres = spark.table(f"{silver_schema}.tb_generos")
df_dim_genres = df_silver_genres.select(trim(col("nome_genero")).alias("nome_genero")) \
    .filter(col("nome_genero").isNotNull() & (col("nome_genero") != "")) \
    .dropDuplicates(["nome_genero"]) \
    .select(
        row_number().over(Window.orderBy("nome_genero")).cast("bigint").alias("sk_genre_id"),
        col("nome_genero")
    )

df_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_genres")

# 3. dim_people (usando nome_entidade e tipo_entidade)
df_silver_people = spark.table(f"{silver_schema}.tb_pessoas_empresas")
df_dim_people = df_silver_people.select(
        col("nome_entidade").alias("nome_pessoa"),
        col("tipo_entidade").alias("tipo_pessoa")
    ) \
    .filter(col("nome_pessoa").isNotNull() & col("tipo_pessoa").isin(['Ator', 'Diretor', 'Roteirista'])) \
    .dropDuplicates(["nome_pessoa", "tipo_pessoa"]) \
    .select(
        row_number().over(Window.orderBy("nome_pessoa", "tipo_pessoa")).cast("bigint").alias("sk_person_id"),
        col("nome_pessoa"),
        col("tipo_pessoa")
    )

df_dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_people")

# 4. dim_companies (filtrando produtoras/estúdios)
df_dim_companies = df_silver_people \
    .filter(~col("tipo_entidade").isin(['Ator', 'Diretor', 'Roteirista'])) \
    .select(trim(col("nome_entidade")).alias("nome_produtora")) \
    .filter(col("nome_produtora").isNotNull() & (col("nome_produtora") != "")) \
    .dropDuplicates(["nome_produtora"]) \
    .select(
        row_number().over(Window.orderBy("nome_produtora")).cast("bigint").alias("sk_company_id"),
        col("nome_produtora")
    )

df_dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_companies")

# 5. dim_reviews
df_silver_reviews = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
col_id_rev = "id_filme" if "id_filme" in df_silver_reviews.columns else "id"

df_reviews_agg = df_silver_reviews.groupBy(col_id_rev).agg(
    count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
    round(avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")
).withColumnRenamed(col_id_rev, "id_filme")

df_dim_reviews = df_reviews_agg.join(df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner") \
    .select(
        row_number().over(Window.orderBy("sk_movie_id")).cast("bigint").alias("sk_review_id"),
        col("sk_movie_id"),
        col("qtd_avaliacoes_usuarios"),
        col("nota_media_usuarios")
    )

df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_reviews")

print("[OK] Todas as dimensões foram criadas com sucesso na Gold!")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] Todas as dimensões foram criadas com sucesso na Gold!


In [0]:
# Verificação rápida das dimensões salvas
display(spark.table("workspace.cinedata_gold.dim_movies").limit(5))
display(spark.table("workspace.cinedata_gold.dim_genres").limit(5))

sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
1,14564,Rings,2017-02-01,2017,102,en,Lançado,"\Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die."
2,32471,Mixtape,2021-12-03,2021,94,en,Lançado,null
3,38258,Grizzly II: Revenge,2020-02-17,2020,74,en,Lançado,\All hell breaks loose when a giant grizzly
4,38492,Billy Joel - Live at Yankee Stadium,2022-06-22,2022,86,en,Lançado,Billy Joel plays his greatest hits in the Big Apple.
5,38700,Bad Boys for Life,2020-01-15,2020,124,en,Lançado,"Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel."


sk_genre_id,nome_genero
1,A Bond Develops Between The Young Man And The Veteran
2,A Bored Housewife
3,A Boy With The Most Unlikely Name
4,A Comedy Dynamics Original
5,A Comedy Of Sex


---
## 2️. Construção das Tabelas Ponte (Bridge Tables)

Como um filme pode ter múltiplos gêneros, vários atores e ser produzido por mais de um estúdio (relacionamento *Muitos-para-Muitos* / N:N), o uso de **Tabelas Ponte** é indispensável. 

Elas conectam a `dim_movies` às dimensões periféricas conectando suas respectivas Surrogate Keys (`sk_movie_id` com `sk_genre_id`, `sk_person_id` e `sk_company_id`), **evitando a duplicação do grão da Tabela Fato**.

In [0]:
# 1. bridge_movie_genre
df_bridge_genre = df_silver_genres \
    .join(df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner") \
    .join(df_dim_genres, on="nome_genero", how="inner") \
    .select("sk_movie_id", "sk_genre_id") \
    .dropDuplicates()

df_bridge_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_genre")

# 2. bridge_movie_person
df_bridge_person = df_silver_people \
    .filter(col("nome_entidade").isNotNull()) \
    .select(
        col("id_filme"),
        col("nome_entidade").alias("nome_pessoa"),
        col("tipo_entidade").alias("tipo_pessoa")
    ) \
    .join(df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner") \
    .join(df_dim_people, on=["nome_pessoa", "tipo_pessoa"], how="inner") \
    .select("sk_movie_id", "sk_person_id") \
    .dropDuplicates()

df_bridge_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_person")

# 3. bridge_movie_company
df_bridge_company = df_silver_people \
    .filter(~col("tipo_entidade").isin(['Ator', 'Diretor', 'Roteirista'])) \
    .select(
        col("id_filme"),
        trim(col("nome_entidade")).alias("nome_produtora")
    ) \
    .filter(col("nome_produtora").isNotNull() & (col("nome_produtora") != "")) \
    .join(df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner") \
    .join(df_dim_companies, on="nome_produtora", how="inner") \
    .select("sk_movie_id", "sk_company_id") \
    .dropDuplicates()

df_bridge_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_company")

print("[OK] Tabelas Ponte construídas com sucesso!")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] Tabelas Ponte construídas com sucesso!


---
## 3️. Construção da Tabela Fato (`fact_movies_performance`)

A tabela **Fato** centraliza os indicadores quantitativos de desempenho da obra.
- **Grão:** Estritamente 1 registro por filme (`sk_movie_id`).
- **Conversão Financeira:** Relaciona a data de lançamento de cada filme (`data_lancamento`) com a série temporal diária de câmbio da Silver (`tb_cotacao_dolar`), efetuando a conversão dos valores monetários de USD para BRL na cotação histórica exata do lançamento.
- **Engajamento:** Consolidação dos índices de popularidade, votos e avaliações médias (TMDB e IMDB).

In [0]:
df_financials = spark.table(f"{silver_schema}.tb_financeiro_filmes")
df_metrics = spark.table(f"{silver_schema}.tb_metricas_engajamento")
df_cotacao = spark.table(f"{silver_schema}.tb_cotacao_dolar")

# 1. Identificação dinâmica do ID e das colunas financeiras
col_id_fin = "id_filme" if "id_filme" in df_financials.columns else "id"
col_id_met = "id_filme" if "id_filme" in df_metrics.columns else "id"

cols_fin = df_financials.columns
col_orc = next((c for c in ["orcamento", "orcamento_usd", "budget"] if c in cols_fin), None)
col_rec = next((c for c in ["receita", "receita_usd", "revenue"] if c in cols_fin), None)

# 2. Obter a cotação histórica da data de lançamento
df_movies_dated = df_dim_movies.select("sk_movie_id", "id_filme", "data_lancamento")
df_movies_with_cotacao = df_movies_dated \
    .join(df_cotacao, df_movies_dated.data_lancamento == df_cotacao.data_cotacao, how="left") \
    .select("sk_movie_id", "id_filme", col("valor_cotacao").alias("cotacao_dolar"))

# 3. Mapeamento e cálculo financeiro em USD e BRL
if col_orc and col_rec:
    df_fato_financials = df_financials.withColumnRenamed(col_id_fin, "id_filme") \
        .join(df_movies_with_cotacao, on="id_filme", how="inner") \
        .select(
            "sk_movie_id",
            col(col_orc).cast("decimal(18,2)").alias("orcamento_usd"),
            col(col_rec).cast("decimal(18,2)").alias("receita_usd"),
            (col(col_rec) - col(col_orc)).cast("decimal(18,2)").alias("lucro_usd"),
            (col(col_orc) * col("cotacao_dolar")).cast("decimal(18,2)").alias("orcamento_brl"),
            (col(col_rec) * col("cotacao_dolar")).cast("decimal(18,2)").alias("receita_brl"),
            ((col(col_rec) - col(col_orc)) * col("cotacao_dolar")).cast("decimal(18,2)").alias("lucro_brl")
        )
else:
    # Caso as colunas já venham com a nomenclatura pronta da Silver
    df_fato_financials = df_financials.withColumnRenamed(col_id_fin, "id_filme") \
        .join(df_movies_with_cotacao, on="id_filme", how="inner") \
        .select(
            "sk_movie_id",
            coalesce(col("orcamento_usd"), col("orcamento")).cast("decimal(18,2)").alias("orcamento_usd"),
            coalesce(col("receita_usd"), col("receita")).cast("decimal(18,2)").alias("receita_usd"),
            coalesce(col("lucro_usd"), (col("receita_usd") - col("orcamento_usd"))).cast("decimal(18,2)").alias("lucro_usd"),
            coalesce(col("orcamento_brl"), (col("orcamento_usd") * col("cotacao_dolar"))).cast("decimal(18,2)").alias("orcamento_brl"),
            coalesce(col("receita_brl"), (col("receita_usd") * col("cotacao_dolar"))).cast("decimal(18,2)").alias("receita_brl"),
            coalesce(col("lucro_brl"), (col("lucro_usd") * col("cotacao_dolar"))).cast("decimal(18,2)").alias("lucro_brl")
        )

# 4. Consolidação com Métricas de Engajamento
df_metrics_renamed = df_metrics.withColumnRenamed(col_id_met, "id_filme")

df_fato_completa = df_fato_financials \
    .join(df_dim_movies.select("sk_movie_id", "id_filme"), on="sk_movie_id", how="inner") \
    .join(df_metrics_renamed, on="id_filme", how="left") \
    .select(
        df_fato_financials["sk_movie_id"],
        "orcamento_usd", "receita_usd", "lucro_usd",
        "orcamento_brl", "receita_brl", "lucro_brl",
        col("popularidade").cast("double"),
        col("nota_media_tmdb").cast("double"),
        col("qtd_votos_tmdb").cast("int"),
        col("nota_media_imdb").cast("double"),
        col("qtd_votos_imdb").cast("int")
    ).dropDuplicates(["sk_movie_id"])

# 5. Salvar Tabela Fato no Unity Catalog
df_fato_completa.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.fact_movies_performance")

print("[OK] Tabela Fato 'fact_movies_performance' gerada com sucesso!")
display(spark.table(f"{gold_schema}.fact_movies_performance").limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[OK] Tabela Fato 'fact_movies_performance' gerada com sucesso!


sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1,25000000.00,83080890.00,58080890.00,78682500.00,261480485.10,182797985.10,24.584,4.97,2375,null,46286
2,null,null,null,null,null,null,8.929,7.06,118,6.6,4617
3,null,null,null,null,null,null,null,3.16,28,null,null
4,null,null,null,null,null,null,2.98,7.1,null,7.8,212
5,null,426505244.00,null,null,1774944223.43,null,46.619,7.14,7570,6.5,199420


---
## 4️. Entrega 2 — Construção da Tabela de Contexto GenAI (`gold_genai_movies_context`)

Esta tabela é projetada especificamente para alimentar o **Vector Search (RAG)** do time de Inteligência Artificial.

### 🛡️ Tratamento da "Casca de Banana" (Valores Nulos)
Para evitar que a concatenação de strings resulte em `NULL` silencioso devido a campos ausentes:
- Utilizou-se a função `coalesce()` para fornecer *fallbacks* padrão para orçamento, receita, diretores, atores e sinopse.
- Utilizou-se `concat_ws()` para criar a frase final corrida e agregar os múltiplos atores de um mesmo filme em uma única string legível.

In [0]:
from pyspark.sql.functions import col, coalesce, concat_ws, collect_set, format_number, lit, when

gold_schema = "workspace.cinedata_gold"

# 1. Carregar as tabelas da Camada Gold
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")
df_fact = spark.table(f"{gold_schema}.fact_movies_performance")
df_bridge_person = spark.table(f"{gold_schema}.bridge_movie_person")
df_dim_people = spark.table(f"{gold_schema}.dim_people")

# 2. Agregar Diretores por filme
df_directors = df_bridge_person \
    .join(df_dim_people.filter(col("tipo_pessoa") == "Diretor"), on="sk_person_id", how="inner") \
    .groupBy("sk_movie_id") \
    .agg(concat_ws(", ", collect_set("nome_pessoa")).alias("diretores"))

# 3. Agregar Atores Principais por filme
df_actors = df_bridge_person \
    .join(df_dim_people.filter(col("tipo_pessoa") == "Ator"), on="sk_person_id", how="inner") \
    .groupBy("sk_movie_id") \
    .agg(concat_ws(", ", collect_set("nome_pessoa")).alias("atores_principais"))

# 4. Consolidar informações com tratamento seguro contra NULLs
df_context_base = df_dim_movies.alias("m") \
    .join(df_fact.alias("f"), on="sk_movie_id", how="left") \
    .join(df_directors.alias("d"), on="sk_movie_id", how="left") \
    .join(df_actors.alias("a"), on="sk_movie_id", how="left") \
    .select(
        col("m.id_filme").alias("movie_id"),
        col("m.titulo").alias("title"),
        
        coalesce(col("m.titulo"), lit("Título não informado")).alias("v_titulo"),
        coalesce(col("m.ano_lancamento").cast("string"), lit("ano não informado")).alias("v_ano"),
        
        # Tratamento correto de valores financeiros nulos
        when(col("f.receita_usd").isNotNull(), concat_ws(" ", lit("USD"), format_number(col("f.receita_usd"), 2)))
            .otherwise(lit("valor não informado")).alias("v_receita"),
            
        when(col("f.orcamento_usd").isNotNull(), concat_ws(" ", lit("USD"), format_number(col("f.orcamento_usd"), 2)))
            .otherwise(lit("valor não informado")).alias("v_orcamento"),
        
        coalesce(col("a.atores_principais"), lit("elenco não informado")).alias("v_atores"),
        coalesce(col("d.diretores"), lit("diretor não informado")).alias("v_diretor"),
        coalesce(col("m.sinopse"), lit("Sem sinopse disponível.")).alias("v_overview")
    )

# 5. Construção da coluna llm_context_document
df_genai_context = df_context_base.select(
    col("movie_id"),
    col("title"),
    concat_ws(
        " ",
        lit("O filme"),
        concat_ws("", col("v_titulo"), lit(",")),
        lit("lançado no ano de"),
        concat_ws("", col("v_ano"), lit(",")),
        lit("faturou"),
        col("v_receita"),
        lit("e teve um custo de"),
        concat_ws("", col("v_orcamento"), lit(".")),
        lit("Estrelado por"),
        col("v_atores"),
        lit("e dirigido por"),
        concat_ws("", col("v_diretor"), lit(",")),
        lit("o filme possui a seguinte sinopse:"),
        col("v_overview")
    ).alias("llm_context_document")
)

# 6. Salvar na Camada Gold
df_genai_context.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.gold_genai_movies_context")

print("[OK] Tabela 'gold_genai_movies_context' atualizada com sucesso!")

display(spark.table(f"{gold_schema}.gold_genai_movies_context").limit(3))

[OK] Tabela 'gold_genai_movies_context' atualizada com sucesso!


movie_id,title,llm_context_document
14564,Rings,"O filme Rings, lançado no ano de 2017, faturou USD 83,080,890.00 e teve um custo de USD 25,000,000.00. Estrelado por Chuck David Willis, Zach Roerig, Laura Slade Wiggins, Alex Roe, Bonnie Morgan, Johnny Galecki, Patrick Walker, Matilda Lutz, Vincent D'onofrio, Aimee Teegarden e dirigido por F. Javier Gutiérrez, o filme possui a seguinte sinopse: \Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die."
32471,Mixtape,"O filme Mixtape, lançado no ano de 2021, faturou valor não informado e teve um custo de valor não informado. Estrelado por Audrey Hsieh, Anthony Timpano, Lucas Yao, Jackson Rathbone, Julie Bowen, Kiefer O'reilly, Olga Petsa, Nick Thune, Gemma Brooke Allen e dirigido por Valerie Weiss, o filme possui a seguinte sinopse: Sem sinopse disponível."
38258,Grizzly II: Revenge,"O filme Grizzly II: Revenge, lançado no ano de 2020, faturou valor não informado e teve um custo de valor não informado. Estrelado por 2.6 e dirigido por English, o filme possui a seguinte sinopse: \All hell breaks loose when a giant grizzly"
